In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip

--2026-01-07 21:24:15--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  5.31MB/s    in 0.9s    

2026-01-07 21:24:16 (5.31 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]



In [3]:
import pandas as pd

ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [4]:
movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [5]:
user_map = {u: i for i, u in enumerate(ratings.user_id.unique())}
movie_map = {m: i for i, m in enumerate(ratings.movie_id.unique())}

ratings["u"] = ratings.user_id.map(user_map)
ratings["i"] = ratings.movie_id.map(movie_map)

n_users = len(user_map)
n_items = len(movie_map)

ratings.head()

,user_id,movie_id,rating,timestamp,u,i
0,196,242,3,881250949,0,0
1,186,302,3,891717742,1,1
2,22,377,1,878887116,2,2
3,244,51,2,880606923,3,3
4,166,346,1,886397596,4,4


In [6]:
from sklearn.model_selection import train_test_split

ratings = ratings.sort_values("timestamp")
train, test = train_test_split(ratings, test_size=0.2, shuffle=False)

In [7]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        # Embeddings for users and items
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)

        # Bias terms
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        # Global mean (learnable optional)
        self.mu = nn.Parameter(torch.tensor(0.0))

        # Initialization
        nn.init.normal_(self.user_factors.weight, std=0.01)
        nn.init.normal_(self.item_factors.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        # user, item are tensors of indices
        pred = self.mu + \
               self.user_bias(user).squeeze() + \
               self.item_bias(item).squeeze() + \
               (self.user_factors(user) * self.item_factors(item)).sum(dim=1)
        return pred

In [8]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 1024
epochs = 20
lr = 0.01
reg = 0.1  # weight decay for regularization

train_users = torch.tensor(train['u'].values, dtype=torch.long)
train_items = torch.tensor(train['i'].values, dtype=torch.long)

print(train_users)
print(train_items)

train_ratings = torch.tensor(train['rating'].values, dtype=torch.float32)

train_dataset = TensorDataset(train_users, train_items, train_ratings)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = MatrixFactorization(n_users, n_items, n_factors=32)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.MSELoss()

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for u, i, r in train_loader:
        optimizer.zero_grad()
        pred = model(u, i)
        loss = loss_fn(pred, r)
        # L2 regularization
        l2_reg = reg * (model.user_factors(u).pow(2).sum() + model.item_factors(i).pow(2).sum())
        loss += l2_reg
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * u.size(0)

    rmse = (total_loss / len(train))**0.5 # Use len(train) here
    print(f"Epoch {epoch+1}, RMSE: {rmse:.4f}")

tensor([101, 101, 101,  ..., 269, 269, 269])
tensor([ 177,  289,  153,  ...,  497,  609, 1166])
Epoch 1, RMSE: 2.9079
Epoch 2, RMSE: 1.6436
Epoch 3, RMSE: 1.1454
Epoch 4, RMSE: 1.0256
Epoch 5, RMSE: 0.9952
Epoch 6, RMSE: 0.9874
Epoch 7, RMSE: 0.9870
Epoch 8, RMSE: 0.9887
Epoch 9, RMSE: 0.9906
Epoch 10, RMSE: 0.9857
Epoch 11, RMSE: 0.9885
Epoch 12, RMSE: 0.9787
Epoch 13, RMSE: 0.9783
Epoch 14, RMSE: 0.9684
Epoch 15, RMSE: 0.9708
Epoch 16, RMSE: 0.9681
Epoch 17, RMSE: 0.9648
Epoch 18, RMSE: 0.9667
Epoch 19, RMSE: 0.9641
Epoch 20, RMSE: 0.9633


In [9]:
model.eval()

test_users = torch.tensor(test['u'].values, dtype=torch.long)
test_items = torch.tensor(test['i'].values, dtype=torch.long)
test_ratings = torch.tensor(test['rating'].values, dtype=torch.float32)

with torch.no_grad():
    pred = model(test_users, test_items)
    rmse = ((pred - test_ratings)**2).mean().sqrt()
    print("Test RMSE:", rmse.item())

Test RMSE: 1.4005690813064575


In [10]:
test_users = torch.tensor([1], dtype=torch.long)
test_items = torch.tensor([50], dtype=torch.long)

with torch.no_grad():
    pred = model(test_users, test_items)
    print("Predicted Rating:", pred.item())

Predicted Rating: 3.891920328140259


In [11]:
def predict_movie(user_id, movie_title):
    user_id_tensor = torch.tensor([user_map[user_id]], dtype=torch.long)

    movie_id = movies[movies['title'].str.contains('Star Wars', case=False, na=False)].iloc[0]["movie_id"]

    movie_id_tensor = torch.tensor([int(movie_id)], dtype=torch.long)

    with torch.no_grad():
        prediction = model(user_id_tensor, movie_id_tensor)
        print(f"Movie Rec Score for {user_id} for {movie_id}: {prediction.item()}")
        return prediction.item()



In [12]:
predict_movie(1, "Star Wars (1977)")

Movie Rec Score for 1 for 50: 3.8201756477355957


3.8201756477355957

In [13]:
movie_pop = train.groupby("movie_id").size().rename("movie_popularity")

# Movie mean rating
movie_mean = train.groupby("movie_id")["rating"].mean().rename("movie_mean_rating")

# User activity
user_activity = train.groupby("user_id").size().rename("user_activity")

df = train.merge(movie_pop, on="movie_id")
df = df.merge(movie_mean, on="movie_id")
df = df.merge(user_activity, on="user_id")

In [14]:
# Access user embeddings
user_embeddings = model.user_factors.weight.detach().cpu().numpy()
print(f"Shape of user embeddings: {user_embeddings.shape}")
print("First 5 user embeddings (first user):")
print(user_embeddings[:5, :5]) # Print a slice for brevity

# Access item embeddings
item_embeddings = model.item_factors.weight.detach().cpu().numpy()
print(f"\nShape of item embeddings: {item_embeddings.shape}")
print("First 5 item embeddings (first item):")
print(item_embeddings[:5, :5]) # Print a slice for brevity

Shape of user embeddings: (943, 32)
First 5 user embeddings (first user):
[[-9.78571989e-05  1.00384816e-04  1.98749876e-05  1.18579283e-05
  -1.08039516e-04]
 [-2.59936496e-06  8.22517322e-05 -5.59870532e-05 -2.45674956e-03
  -1.93117849e-05]
 [-3.74101364e-05  2.64141909e-05 -1.15434159e-04 -3.68286390e-04
   4.64126279e-05]
 [-1.07520702e-03 -2.03166550e-04  1.63970471e-05 -4.80778726e-05
  -2.51676393e-05]
 [-2.01456351e-05 -9.59933368e-06 -2.03028667e-05  5.97829967e-05
  -3.63292565e-05]]

Shape of item embeddings: (1682, 32)
First 5 item embeddings (first item):
[[-1.2021880e-03 -6.8981707e-04  1.7379243e-03  5.7356628e-03
  -2.3495153e-02]
 [ 4.7665625e-04 -9.5128380e-05 -4.1657713e-06 -1.2124328e-05
   7.6542980e-05]
 [ 6.4369533e-06 -8.7289044e-07 -2.4460192e-04 -1.2923587e-06
   2.5172598e-05]
 [ 1.0780668e-03  4.3004453e-05 -9.9460922e-06  1.1949920e-05
   1.2799568e-04]
 [ 1.3565953e-04  9.4003498e-04 -1.2054278e-04  3.3068216e-05
  -9.8815118e-04]]


In [15]:
import numpy as np

def dot(u, i):
    return np.dot(u, i)


df["mf_score"] = df.apply(
    lambda x: dot(user_embeddings[int(x.user_id) - 1], item_embeddings[int(x.movie_id) - 1]),
    axis=1
)

In [16]:
feature_cols = [
    "mf_score",
    "movie_popularity",
    "movie_mean_rating",
    "user_activity"
]

X = df[feature_cols]
y = df["rating"]

print(X)

           mf_score  movie_popularity  movie_mean_rating  user_activity
0     -1.081411e-06               139           3.402878             46
1     -4.025922e-05               370           3.686486             46
2     -1.099767e-07               162           3.814815             46
3     -4.348039e-04               192           4.083333             46
4      1.804922e-05               266           4.233083             46
...             ...               ...                ...            ...
79995  4.628213e-07               150           3.253333             32
79996 -5.492476e-07               183           3.071038             32
79997  2.648321e-06               101           3.148515             32
79998 -8.271813e-08                 9           3.000000             32
79999 -4.761262e-09                17           2.941176             32

[80000 rows x 4 columns]


In [17]:
import xgboost as xgb

group_sizes = df.groupby("user_id").size().to_list()

dtrain = xgb.DMatrix(X, label=y)
dtrain.set_group(group_sizes)

params = {
    "objective": "rank:pairwise",   # or "rank:ndcg"
    "eval_metric": "map",
    "eta": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8
}

xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=100
)

In [18]:
def recommend(user_id, candidate_movies):
    rows = []

    for movie_id in candidate_movies:
        rows.append({
            "mf_score": dot(user_embeddings[user_id], item_embeddings[movie_id]),
            "movie_popularity": movie_pop[movie_id],
            "movie_mean_rating": movie_mean[movie_id],
            "user_activity": user_activity[user_id]
        })

    X_cand = pd.DataFrame(rows)
    scores = xgb_model.predict(xgb.DMatrix(X_cand))

    movie_titles = movies[movies["movie_id"].isin(candidate_movies)]["title"].values

    ranked = sorted(zip(movie_titles, scores), key=lambda x: x[1], reverse=True)
    return ranked[:10]


recommend(1, [5,10,15,20,12,234,352, 543,234])

[('Angels and Insects (1995)', np.float32(0.70020694)),
 ('Jaws (1975)', np.float32(-0.07144595)),
 ('Usual Suspects, The (1995)', np.float32(-0.08179233)),
 ('Misérables, Les (1995)', np.float32(-0.09788528)),
 ('Richard III (1995)', np.float32(-0.10392461)),
 ('Copycat (1995)', np.float32(-0.5111142)),
 ("Mr. Holland's Opus (1995)", np.float32(-0.5826055)),
 ('Spice World (1997)', np.float32(-1.6232548))]

##Test Data Training

In [19]:
movie_pop = test.groupby("movie_id").size().rename("movie_popularity")

# Movie mean rating
movie_mean = test.groupby("movie_id")["rating"].mean().rename("movie_mean_rating")

# User activity
user_activity = test.groupby("user_id").size().rename("user_activity")

df_test = test.merge(movie_pop, on="movie_id")
df_test = df_test.merge(movie_mean, on="movie_id")
df_test = df_test.merge(user_activity, on="user_id")
df_test

,user_id,movie_id,rating,timestamp,u,i,movie_popularity,movie_mean_rating,user_activity
0,3,322,3,889237269,269,51,57,3.017544,22
1,3,335,1,889237269,269,620,4,2.250000,22
2,3,323,2,889237269,269,144,52,2.846154,22
3,3,325,1,889237297,269,331,20,2.450000,22
4,3,264,2,889237297,269,116,21,3.285714,22
...,...,...,...,...,...,...,...,...,...
19995,729,748,4,893286638,724,175,66,3.212121,21
19996,729,272,4,893286638,724,130,99,4.252525,21
19997,729,300,4,893286638,724,652,116,3.603448,21
19998,729,689,4,893286638,724,890,26,3.346154,21


In [22]:
import numpy as np

def dot(u, i):
    return np.dot(u, i)


df_test["mf_score"] = df_test.apply(
    lambda x: dot(user_embeddings[int(x.user_id) - 1], item_embeddings[int(x.movie_id) - 1]),
    axis=1
)

In [23]:
df_test

,user_id,movie_id,rating,timestamp,u,i,movie_popularity,movie_mean_rating,user_activity,mf_score
0,3,322,3,889237269,269,51,57,3.017544,22,4.851465e-07
1,3,335,1,889237269,269,620,4,2.250000,22,-4.611214e-05
2,3,323,2,889237269,269,144,52,2.846154,22,-2.405274e-07
3,3,325,1,889237297,269,331,20,2.450000,22,-2.145912e-08
4,3,264,2,889237297,269,116,21,3.285714,22,-7.080619e-08
...,...,...,...,...,...,...,...,...,...,...
19995,729,748,4,893286638,724,175,66,3.212121,21,4.827305e-06
19996,729,272,4,893286638,724,130,99,4.252525,21,2.211482e-07
19997,729,300,4,893286638,724,652,116,3.603448,21,-1.477963e-06
19998,729,689,4,893286638,724,890,26,3.346154,21,-1.243136e-05


In [24]:
feature_cols = [
    "mf_score",
    "movie_popularity",
    "movie_mean_rating",
    "user_activity"
]

X_test = df_test[feature_cols]
y_test = df_test["rating"]

           mf_score  movie_popularity  movie_mean_rating  user_activity
0      4.851465e-07                57           3.017544             22
1     -4.611214e-05                 4           2.250000             22
2     -2.405274e-07                52           2.846154             22
3     -2.145912e-08                20           2.450000             22
4     -7.080619e-08                21           3.285714             22
...             ...               ...                ...            ...
19995  4.827305e-06                66           3.212121             21
19996  2.211482e-07                99           4.252525             21
19997 -1.477963e-06               116           3.603448             21
19998 -1.243136e-05                26           3.346154             21
19999  3.599460e-06               149           4.046980             21

[20000 rows x 4 columns]


In [34]:
def recall_at_k(
    df_test,
    xgb_model,
    feature_cols,
    k=10,
    relevance_threshold=4
):
    recalls = []

    for user_id, user_df in df_test.groupby("user_id"):
        # Ground truth relevant items
        relevant_items = user_df[
            user_df["rating"] >= relevance_threshold
        ]["movie_id"].values

        if len(relevant_items) == 0:
            continue  # skip users with no positives

        # Predict scores
        X_user = user_df[feature_cols]
        duser = xgb.DMatrix(X_user)
        scores = xgb_model.predict(duser)

        user_df = user_df.copy()
        user_df["score"] = scores

        # Top-K recommended movies
        top_k_items = (
            user_df
            .sort_values("score", ascending=False)
            .head(k)["movie_id"]
            .values
        )

        # Recall calculation
        hits = len(set(top_k_items) & set(relevant_items))
        recall = hits / len(relevant_items)
        recalls.append(recall)

    return np.mean(recalls)


In [38]:
feature_cols = [
    "mf_score",
    "movie_popularity",
    "movie_mean_rating",
    "user_activity"
]

recall_10 = recall_at_k(
    df_test=df_test,
    xgb_model=xgb_model,
    feature_cols=feature_cols,
    k=10,
    relevance_threshold=5
)

print(f"Recall@10: {recall_10:.4f}")

Recall@10: 0.4589
